# Vlastní experimenty 1: slice sampling vs. Random Walk Metropolis v 1D

Tento notebook navazuje na `motivation.ipynb` a nabízí možnosti, jak postupovat v první fázi práce: vlastní implementace obou metod, diagnostika MCMC a první systematické srovnání. Není to pevné zadání – pokud máte vlastní nápad, jak něco otestovat, klidně se od něj odchylte; podstatné je, aby každý experiment končil **tabulkou nebo grafem a krátkým písemným závěrem** (několik vět opřených o konkrétní čísla), protože přesně z takových závěrů se později skládá praktická část práce.

**Doporučení**
- Kód v experimentech 1, 2, 4 a 6 zkuste napsat **bez nahlížení do `samplers.py`** (po dokončení si ho s referencí porovnejte).
- Všude používejte seedovaný generátor `np.random.default_rng(seed)`, aby šly výsledky zopakovat.
- Srovnáváme **počtem vyhodnocení cílové hustoty**, ne časem.
- Co nestihnete, nechte být – lepší tři dokončené experimenty se závěry než šest rozpracovaných.

**Literatura:** 
- Neal (2003), Single-variable slice sampling, procedury "stepping-out", "doubling", "shrinkage" - kapitola 4 od začátku po sekci 4.2 včetně
https://www.jstor.org/stable/pdf/3448413.pdf
- Robert & Casella (2004), RWMH algoritmus - sekce 7.5
https://mcube.lab.nycu.edu.tw/~cfung/docs/books/robert2004monte_carlo_statistical_methods.pdf
- k odhadu autokorelačního času Foreman-Mackey et al. (2013), *emcee: The MCMC Hammer* - doporučuji tutoriál
https://emcee.readthedocs.io/en/stable/tutorials/autocorr/
(případně článek https://arxiv.org/pdf/1202.3665)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from samplers import slice_sampling, metropolis_hastings   # referenční implementace (jen pro kontrolu a pro experimenty 3 a 5)

SQRT_2PI = np.sqrt(2 * np.pi)

def normal_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * SQRT_2PI)

def make_mixture(m=2.0):
    """Směs 0,3·N(-m, 1²) + 0,7·N(m, 0,7²); m řídí vzdálenost módů. Pro m=2 jde o rozdělení z motivace.
    Vrací (log_pdf, true_mean, true_sd, true_P_left), kde true_P_left = P(x < 0)."""
    from math import erf, sqrt
    Phi = lambda z: 0.5 * (1 + erf(z / sqrt(2)))
    def log_pdf(x):
        p = 0.3 * normal_pdf(x, -m, 1.0) + 0.7 * normal_pdf(x, m, 0.7)
        return np.log(p) if p > 0 else -np.inf
    mean = 0.3 * (-m) + 0.7 * m
    var = 0.3 * (1 + m**2) + 0.7 * (0.49 + m**2) - mean**2
    p_left = 0.3 * Phi(m / 1.0) + 0.7 * Phi(-m / 0.7)
    return log_pdf, mean, np.sqrt(var), p_left

def log_std_normal(x):
    return -0.5 * x**2

def iact_ref(x, c=5.0, max_lag=None):
    """Referenční integrovaný autokorelační čas tau = 1 + 2 * sum_{T>=1} rho(T)
    (Foreman-Mackey et al. 2013, rov. (12)-(13)). Součet je useknut Sokalovým automatickým oknem,
    jak je implementováno v balíku acor / emcee (Goodman & Weare 2010): vezme se nejmenší M,
    pro které M >= c * tau(M), kde tau(M) = 1 + 2 * sum_{T=1}^{M} rho(T); výchozí c = 5.
    Použijte jen pro kontrolu vlastní implementace v experimentu 2."""
    x = np.asarray(x, dtype=float) - np.mean(x)
    n = len(x)
    max_lag = max_lag or n // 2
    f = np.fft.rfft(x, 2 * n)
    rho = np.fft.irfft(f * np.conj(f))[:max_lag]
    rho = rho / rho[0]
    taus = 1.0 + 2.0 * np.cumsum(rho[1:])          # taus[M-1] = tau(M)
    M = np.arange(1, len(taus) + 1)
    ok = np.nonzero(M >= c * taus)[0]
    if len(ok) == 0:
        return taus[-1]                               # okno se nenašlo – řetězec je příliš krátký vzhledem k tau
    return taus[ok[0]]

def print_table(headers, rows, fmt=None):
    """Vytiskne tabulku v Markdown syntaxi (lze zkopírovat do textu práce)."""
    fmt = fmt or ["{}"] * len(headers)
    print("| " + " | ".join(headers) + " |")
    print("|" + "---|" * len(headers))
    for r in rows:
        print("| " + " | ".join(f.format(v) for f, v in zip(fmt, r)) + " |")

log_mix, MIX_MEAN, MIX_SD, MIX_P_LEFT = make_mixture(2.0)
print(f"Směs z motivace: E[x] = {MIX_MEAN:.4f}, sd = {MIX_SD:.4f}, P(x<0) = {MIX_P_LEFT:.4f}  (pozor: není to 0,3 – komponenty se překrývají)")

---
## Experiment 1 – Vlastní implementace a test správnosti

Naprogramujte vlastní **Random Walk Metropolis** a **1D slice sampling se stepping-out a shrinking** (Neal 2003, obr. 3 a 5). Obě funkce vracejí pole vzorků a **celkový počet vyhodnocení `log_pdf`** – počítejte každé volání, včetně testů ve stepping-out.

Správnost ověřte na dvou rozděleních: $\mathcal N(0,1)$ a směsi z motivace. Pro každou kombinaci (rozdělení × metoda) spočítejte střední hodnotu, směrodatnou odchylku a $P(x<0)$ a porovnejte s přesnými hodnotami. Odhad je v pořádku, je-li odchylka do cca $3\times$ Monte Carlo chyby; tu zatím hrubě odhadněte jako $\mathrm{sd}/\sqrt{n/20}$ (v experimentu 2 ji spočítáte pořádně přes ESS).

In [ ]:
def my_rwm(log_pdf, x0, n_samples, sigma, rng):
    """Random Walk Metropolis s návrhem x' = x + N(0, sigma²).
    Vrací (samples, accept_rate, n_evals)."""
    # TODO: implementujte; nezapomeňte uchovávat log_pdf(x) aktuálního bodu (1 vyhodnocení na krok)
    raise NotImplementedError("Experiment 1: my_rwm")


def my_slice(log_pdf, x0, n_samples, w, rng):
    """1D slice sampling, stepping-out + shrinking (Neal 2003, Fig. 3 a 5), bez omezení počtu kroků m.
    Vrací (samples, n_evals)."""
    # TODO: implementujte; počítejte každé vyhodnocení log_pdf
    raise NotImplementedError("Experiment 1: my_slice")

In [ ]:
N, BURN = 20_000, 1_000
targets = {
    "N(0,1)": (log_std_normal, 0.0, 1.0, 0.5),
    "směs":   (log_mix, MIX_MEAN, MIX_SD, MIX_P_LEFT),
}
rows = []
for name, (lp, m_true, sd_true, pl_true) in targets.items():
    s_r, acc, ne_r = my_rwm(lp, x0=0.0, n_samples=N, sigma=2.0, rng=np.random.default_rng(1))
    s_s, ne_s = my_slice(lp, x0=0.0, n_samples=N, w=2.0, rng=np.random.default_rng(1))
    for meth, s, ne in [("RWM σ=2", s_r[BURN:], ne_r), ("slice w=2", s_s[BURN:], ne_s)]:
        mc_err = s.std() / np.sqrt(len(s) / 20)
        rows.append([name, meth, s.mean(), m_true, s.std(), sd_true, np.mean(s < 0), pl_true, mc_err, ne / N])

print_table(["rozdělení", "metoda", "E[x] odhad", "E[x] přesně", "sd odhad", "sd přesně", "P(x<0) odhad", "P(x<0) přesně", "~MC chyba E[x]", "vyhodnocení / vzorek"],
            rows, ["{}", "{}", "{:.4f}", "{:.4f}", "{:.4f}", "{:.4f}", "{:.4f}", "{:.4f}", "{:.4f}", "{:.1f}"])

**Kontrola proti referenci:** po dokončení spusťte totéž s `metropolis_hastings` a `slice_sampling` ze `samplers.py` (stejný seed) a porovnejte počty vyhodnocení na vzorek. Shodují se? Pokud ne, kde se vaše implementace liší (např. počítání testů ve stepping-out)?

**Závěr k experimentu 1** *(doplňte)*: Jsou obě implementace správné? Kolik vyhodnocení $f$ stojí jeden vzorek u každé metody a odkud se ten rozdíl bere?

> …

---
## Experiment 2 – Autokorelace, IACT a efektivní počet vzorků

Vzorky MCMC jsou závislé, proto $n$ vzorků neodpovídá $n$ nezávislým. Implementujte:

1. `acf(x, max_lag)` – výběrovou autokorelační funkci $\rho_k = C(k)/C(0)$ pro $k = 0,\dots,$ `max_lag`, kde $C(k)$ je výběrová autokovariance podle rov. (13) ve Foreman-Mackey et al. (2013) (nebo ekvivalentně přes FFT),
2. `iact(x)` – integrovaný autokorelační čas $\tau = 1 + 2\sum_{k\ge 1}\rho_k$ (rov. (12) tamtéž). Součet je nutné někde useknout, jinak ho přehluší šum odhadu $\rho_k$ pro velká $k$. Použijte Sokalovo automatické okno, které používá balík acor/emcee (Goodman & Weare 2010): spočítejte $\tau(M) = 1 + 2\sum_{k=1}^{M}\rho_k$ a vezměte nejmenší $M$, pro něž $M \ge c\,\tau(M)$, s $c = 5$,
3. $\mathrm{ESS} = n/\tau$ a Monte Carlo chybu střední hodnoty $\mathrm{sd}/\sqrt{\mathrm{ESS}}$.

Výsledek ověřte proti `iact_ref` (mělo by vyjít prakticky totéž) a volitelně proti `emcee.autocorr.integrated_time`, která implementuje stejné pravidlo. Rozmyslete, proč je potřeba řetězec aspoň $\sim 50\,\tau$ dlouhý, aby odhad dával smysl.

In [ ]:
def acf(x, max_lag=200):
    """Výběrová autokorelace rho_k, k = 0..max_lag. Vrací pole délky max_lag+1."""
    # TODO
    raise NotImplementedError("Experiment 2: acf")


def iact(x, max_lag=None, c=5.0):
    """Integrovaný autokorelační čas tau = 1 + 2 * sum_{k>=1} rho_k, useknutý Sokalovým oknem (M >= c * tau(M), c = 5).
    max_lag=None znamená len(x) // 2."""
    # TODO
    raise NotImplementedError("Experiment 2: iact")


def ess(x):
    return len(x) / iact(x)

In [ ]:
# Graf ACF a tabulka IACT/ESS pro RWM (σ = 0,1; 2; 40) a slice (w = 0,1; 2; 40) na směsi z motivace
N, BURN = 20_000, 1_000
runs = []
for sigma in [0.1, 2.0, 40.0]:
    s, acc, ne = my_rwm(log_mix, 0.0, N, sigma, np.random.default_rng(0))
    runs.append((f"RWM σ={sigma}", s[BURN:], ne))
for w in [0.1, 2.0, 40.0]:
    s, ne = my_slice(log_mix, 0.0, N, w, np.random.default_rng(0))
    runs.append((f"slice w={w}", s[BURN:], ne))

fig, ax = plt.subplots(figsize=(8, 3.5))
for name, s, ne in runs:
    ax.plot(acf(s, 100), label=name, lw=1.2)
ax.axhline(0, color="k", lw=0.5); ax.set_xlabel("zpoždění $k$"); ax.set_ylabel(r"$\rho_k$"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

rows = []
for name, s, ne in runs:
    tau = iact(s)
    rows.append([name, tau, iact_ref(s), len(s) / tau, s.std() / np.sqrt(len(s) / tau), ne / N])
print_table(["běh", "IACT (vlastní)", "IACT (reference)", "ESS", "MC chyba E[x]", "vyhodnocení / vzorek"],
            rows, ["{}", "{:.1f}", "{:.1f}", "{:.0f}", "{:.4f}", "{:.1f}"])

**Závěr k experimentu 2** *(doplňte)*: Jak moc se liší $n$ a ESS? Který běh má nejhorší míchání a jak to vypadá na ACF? Vraťte se k tabulce z experimentu 1 – byla hrubá MC chyba $\mathrm{sd}/\sqrt{n/20}$ přiměřená?

> …

---
## Experiment 3 – Citlivost na volbu parametru (kritérium 1 ze zadání)

Pro směs z motivace projděte $\sigma$ (RWM) i $w$ (slice) na **logaritmické mřížce** `np.logspace(-1.5, 2.5, 9)` (≈ 0,03 až 300). Pro každou hodnotu spočítejte

- ESS na 1000 vyhodnocení $f$: $\;1000\cdot \mathrm{ESS}/(\text{počet vyhodnocení})$,
- počet přechodů mezi módy (změn znaménka $x$) na 1000 vzorků.

Vykreslete ESS/1000 vyhodnocení jako funkci parametru (log osa $x$) pro obě metody do jednoho grafu a vyplňte tabulku. **Robustnost** kvantifikujte jako šířku rozsahu parametru (v řádech, tj. $\log_{10}(\text{max}/\text{min})$), ve kterém metoda dosahuje alespoň 50 % svého maxima ESS/vyhodnocení.

Zde můžete použít vlastní samplery z experimentu 1, nebo referenční ze `samplers.py` – uveďte které.

In [ ]:
grid = np.logspace(-1.5, 2.5, 9)
N, BURN = 20_000, 1_000

def evaluate_run(samples, n_evals, n_total):
    """Vrací (ESS na 1000 vyhodnocení, přechody mezi módy na 1000 vzorků)."""
    s = samples[BURN:]
    e = len(s) / iact(s)
    switches = np.sum(np.diff(s < 0) != 0)
    return 1000 * e / (n_evals * len(s) / n_total), 1000 * switches / len(s)

res_rwm, res_slice = [], []
for p in grid:
    s, acc, ne = my_rwm(log_mix, 0.0, N, p, np.random.default_rng(0))      # nebo metropolis_hastings(log_mix, 0.0, N, proposal_std=p, rng=...)
    res_rwm.append((p, acc, *evaluate_run(s, ne, N), ne / N))
    s, ne = my_slice(log_mix, 0.0, N, p, np.random.default_rng(0))          # nebo slice_sampling(log_mix, 0.0, N, w=p, rng=...)
    res_slice.append((p, np.nan, *evaluate_run(s, ne, N), ne / N))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(grid, [r[2] for r in res_rwm], "o-", color="coral", label="RWM ($\\sigma$)")
ax.semilogx(grid, [r[2] for r in res_slice], "s-", color="steelblue", label="slice ($w$)")
ax.set_xlabel("parametr ($\\sigma$ resp. $w$)"); ax.set_ylabel("ESS / 1000 vyhodnocení $f$"); ax.legend()
plt.tight_layout(); plt.show()

print("RWM"); print_table(["σ", "akceptace", "ESS/1000 vyh.", "přechody/1000 vz.", "vyhodnocení/vzorek"], res_rwm, ["{:.3g}", "{:.2f}", "{:.1f}", "{:.1f}", "{:.1f}"])
print("\nslice"); print_table(["w", "–", "ESS/1000 vyh.", "přechody/1000 vz.", "vyhodnocení/vzorek"], res_slice, ["{:.3g}", "{}", "{:.1f}", "{:.1f}", "{:.1f}"])

def robust_range(params, eff, frac=0.5):
    """Šířka (v řádech) rozsahu parametru, kde eff >= frac * max(eff)."""
    ok = np.asarray(eff) >= frac * np.max(eff)
    p = np.asarray(params)[ok]
    return np.log10(p.max() / p.min()), p.min(), p.max()

for name, res in [("RWM", res_rwm), ("slice", res_slice)]:
    width, lo, hi = robust_range([r[0] for r in res], [r[2] for r in res])
    print(f"{name:5}: max ESS/1000 vyh. = {max(r[2] for r in res):.1f}; ≥50 % maxima pro parametr v [{lo:.3g}, {hi:.3g}] → šířka {width:.1f} řádu")

**Závěr k experimentu 3** *(doplňte)*: Která metoda má vyšší maximum ESS na vyhodnocení a o kolik? Která má širší „bezpečný“ rozsah parametru? Jak byste jednou větou formulovali kompromis mezi oběma metodami pro někoho, kdo nemá čas ladit? Souvisí akceptační poměr RWM s jeho maximem (srov. Roberts & Rosenthal 2001: optimum ≈ 0,44 v 1D)?

> …

---
## Experiment 4 – Zbytečné vyhodnocení ve slice sampleru

V referenční implementaci `slice_sampling` (a možná i ve vaší) se na začátku každé iterace počítá `log_pdf(x)` aktuálního bodu, ačkoli tato hodnota je známa z posledního kroku shrinking (přijatý bod $x'$ má známé $\log f(x')$).

1. Ověřte to: změřte počet vyhodnocení na vzorek pro $w=2$ a odhadněte, kolik z toho je toto opakované vyhodnocení.
2. Upravte svou implementaci tak, aby hodnotu uchovávala, a změřte znovu.
3. Doplňte do tabulky, o kolik se změní ESS na 1000 vyhodnocení pro $w \in \{0{,}1;\ 2;\ 40\}$.

In [ ]:
def my_slice_cached(log_pdf, x0, n_samples, w, rng):
    """Jako my_slice, ale log_pdf aktuálního bodu se nepočítá znovu v každé iteraci."""
    # TODO
    raise NotImplementedError("Experiment 4: my_slice_cached")

rows = []
for w in [0.1, 2.0, 40.0]:
    s1, ne1 = my_slice(log_mix, 0.0, N, w, np.random.default_rng(0))
    s2, ne2 = my_slice_cached(log_mix, 0.0, N, w, np.random.default_rng(0))
    rows.append([w, ne1 / N, ne2 / N, evaluate_run(s1, ne1, N)[0], evaluate_run(s2, ne2, N)[0]])
print_table(["w", "vyh./vzorek původní", "vyh./vzorek cache", "ESS/1000 vyh. původní", "ESS/1000 vyh. cache"],
            rows, ["{}", "{:.2f}", "{:.2f}", "{:.1f}", "{:.1f}"])

**Závěr k experimentu 4** *(doplňte)*: Dávají obě verze při stejném seedu stejné vzorky? (Měly by – proč?) Jak velký je relativní zisk a pro které $w$ je největší? Mění to závěr experimentu 3?

> …

---
## Experiment 5 – Multimodalita: vzdálenost módů (kritérium 3 ze zadání)

Funkce `make_mixture(m)` vrací směs $0{,}3\,\mathcal N(-m,1^2) + 0{,}7\,\mathcal N(m,0{,}7^2)$. Pro $m \in \{1, 2, 3, 4\}$ spusťte

- RWM s $\sigma$, které vyšlo nejlépe v experimentu 3,
- slice s $w = 2$ a s $w = 20$,

a do tabulky zapište: odhad $P(x<0)$ vs. přesná hodnota, přechody mezi módy na 1000 vzorků, ESS na 1000 vyhodnocení. Použijte delší řetězce (`N = 50_000`), u $m=4$ jsou přechody vzácné.

In [ ]:
N, BURN = 50_000, 2_000
SIGMA_BEST = 2.0   # TODO: dosaďte nejlepší σ z experimentu 3

def rwm_pair(*args):
    s, acc, ne = my_rwm(*args)
    return s, ne          # stejný tvar návratu jako my_slice

rows = []
for m in [1, 2, 3, 4]:
    lp, _, _, p_left = make_mixture(m)
    configs = [
        ("RWM σ=%.3g" % SIGMA_BEST, lambda: rwm_pair(lp, 0.0, N, SIGMA_BEST, np.random.default_rng(0))),
        ("slice w=2",  lambda: my_slice(lp, 0.0, N, 2.0, np.random.default_rng(0))),
        ("slice w=20", lambda: my_slice(lp, 0.0, N, 20.0, np.random.default_rng(0))),
    ]
    for name, run in configs:
        s, ne = run()
        eff, sw = evaluate_run(s, ne, N)
        rows.append([m, name, np.mean(s[BURN:] < 0), p_left, sw, eff])
print_table(["m", "metoda", "P(x<0) odhad", "P(x<0) přesně", "přechody/1000 vz.", "ESS/1000 vyh."],
            rows, ["{}", "{}", "{:.3f}", "{:.3f}", "{:.1f}", "{:.1f}"])

**Závěr k experimentu 5** *(doplňte)*: Od jaké vzdálenosti módů přestává která metoda správně odhadovat $P(x<0)$? Pomáhá slice samplingu větší $w$ a za jakou cenu? Proč u RWM nelze problém vyřešit prostým zvětšením $\sigma$? Co z toho plyne pro posteriory v bayesovské inverzi, které mohou mít více módů?

> …

---
## Experiment 6 (bonus) – Varianta doubling

Neal (2003, obr. 4 a 6) navrhuje místo stepping-out **doubling**: interval se zdvojnásobuje náhodně doleva nebo doprava, dokud oba kraje neleží mimo řez. Shrinking pak vyžaduje dodatečný test přijatelnosti (obr. 6), aby byla zachována reverzibilita.

Implementujte `my_slice_doubling` a porovnejte s `my_slice` (stepping-out) počet vyhodnocení na vzorek a ESS na 1000 vyhodnocení pro $w \in \{0{,}01;\ 0{,}1;\ 1;\ 10\}$.

In [ ]:
def my_slice_doubling(log_pdf, x0, n_samples, w, rng):
    """1D slice sampling s doubling (Neal 2003, Fig. 4) a testem přijatelnosti (Fig. 6). Vrací (samples, n_evals)."""
    # TODO (bonus)
    raise NotImplementedError("Experiment 6: my_slice_doubling")

N, BURN = 20_000, 1_000
rows = []
for w in [0.01, 0.1, 1.0, 10.0]:
    s1, ne1 = my_slice(log_mix, 0.0, N, w, np.random.default_rng(0))
    s2, ne2 = my_slice_doubling(log_mix, 0.0, N, w, np.random.default_rng(0))
    rows.append([w, ne1 / N, ne2 / N, evaluate_run(s1, ne1, N)[0], evaluate_run(s2, ne2, N)[0]])
print_table(["w", "vyh./vzorek stepping-out", "vyh./vzorek doubling", "ESS/1000 vyh. stepping-out", "ESS/1000 vyh. doubling"],
            rows, ["{}", "{:.1f}", "{:.1f}", "{:.1f}", "{:.1f}"])

**Závěr k experimentu 6** *(doplňte)*: Kdy se doubling vyplatí? Je některá varianta „bezpečnější“ při špatně zvoleném $w$?

> …

---
## Celkový závěr *(doplňte, 10–15 vět)*

Shrňte, co z tabulek plyne pro tři kritéria ze zadání – **citlivost na parametry**, **výpočetní náročnost** (vyhodnocení $f$ na efektivní vzorek) a **použitelnost na různé tvary rozdělení** (zde: vzdálenost módů). Napište i to, co z 1D experimentů **nelze** usoudit a co bude třeba otestovat ve více dimenzích a na skutečné inverzní úloze.

> …

**Doporučení na závěr:** notebook s vyplněnými závěry nahrajte do repozitáře na githubu. Hodí se k tomu přidat i jednostránkové shrnutí vlastními slovy, co je slice sampling a proč je to MCMC metoda – bude to základ teoretické kapitoly.